#  GroupDNA — WhatsApp Group Chat Analyzer

**Project:** GroupDNA — Minor Project, The Unlox Academy  
**Dataset:** `hostel_bois.txt` (synthetic, 60 days, 6 participants)  
**Constraints:** Python fundamentals + NumPy only. No pandas, no matplotlib, no regex, no collections.Counter.

---

## Feature 1 — The Chat Parser

In [4]:
from datetime import datetime, date, timedelta
import numpy as np

def parse_chat(filepath):
    """
    Parses a WhatsApp export .txt file.
    Handles: system messages, media-omitted, deleted msgs, multi-line, empty lines.
    Returns: messages list, system_count, media_count, deleted_count.
    """
    messages, prev_entry = [], None
    system_count = media_count = deleted_count = 0

    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for raw in lines:
        line = raw.strip()
        if not line:
            continue
        # Detect DD/MM/YY date stamp at start
        is_new = (len(line) >= 8 and line[2] == "/" and line[5] == "/"
                  and line[:2].isdigit() and line[3:5].isdigit())
        # Multi-line continuation
        if not is_new:
            if prev_entry and prev_entry["type"] == "normal":
                prev_entry["text"] += " " + line
            continue
        parts = line.split(" - ", 1)
        if len(parts) < 2:
            continue
        timestamp_str, rest = parts[0].strip(), parts[1].strip()
        if ": " not in rest:                # system message
            system_count += 1
            continue
        sender, text = rest.split(": ", 1)
        sender, text = sender.strip(), text.strip()
        try:
            dt = datetime.strptime(timestamp_str, "%d/%m/%y, %H:%M")
        except ValueError:
            system_count += 1
            continue
        if text == "<Media omitted>":
            msg_type = "media"; media_count += 1
        elif text == "This message was deleted":
            msg_type = "deleted"; deleted_count += 1
        else:
            msg_type = "normal"
        entry = {"timestamp": timestamp_str, "sender": sender,
                 "text": text, "type": msg_type, "dt": dt}
        messages.append(entry)
        prev_entry = entry

    return messages, system_count, media_count, deleted_count


CHAT_FILE    = "hostel_bois.txt"
messages, system_count, media_count, deleted_count = parse_chat(CHAT_FILE)

PARTICIPANTS = ["Rahul", "Priya", "Aman", "Karan", "Neha", "Vikas"]
PERSON_IDX   = {p: i for i, p in enumerate(PARTICIPANTS)}
normal_msgs  = [m for m in messages if m["type"] == "normal"]
all_dates    = sorted(set(m["dt"].date() for m in normal_msgs))
start_date   = all_dates[0]
end_date     = all_dates[-1]
total_days   = (end_date - start_date).days + 1

calendar = []
d = start_date
while d <= end_date:
    calendar.append(d)
    d += timedelta(days=1)

print(f"Successfully parsed {len(normal_msgs)} messages from "
      f"{len(PARTICIPANTS)} participants over {total_days} days.")
print(f"Skipped: {system_count} system | {media_count} media-omitted | {deleted_count} deleted")
print("\nFirst 5 messages:")
for m in normal_msgs[:5]:
    print(f"  [{m['timestamp']}]  {m['sender']}: {m['text'][:55]}")
print("\nLast 5 messages:")
for m in normal_msgs[-5:]:
    print(f"  [{m['timestamp']}]  {m['sender']}: {m['text'][:55]}")


Successfully parsed 3127 messages from 6 participants over 60 days.
Skipped: 4 system | 32 media-omitted | 15 deleted

First 5 messages:
  [01/04/24, 01:17]  Rahul: scene fix
  [01/04/24, 01:17]  Rahul: haan
  [01/04/24, 01:18]  Rahul: kya scene
  [01/04/24, 02:13]  Rahul: abhi free hai?
  [01/04/24, 02:13]  Rahul: abey

Last 5 messages:
  [30/05/24, 19:14]  Priya: Take care everyone
  [30/05/24, 19:28]  Priya: Karan that sounds tough, take care
  [30/05/24, 21:17]  Aman: the existential dread is back
  [30/05/24, 21:30]  Karan: Long day guys, woke up at six for that placement worksh
  [30/05/24, 23:31]  Aman: anyone awake?


## Feature 2 — Group Overview & Per-Person Stats

In [5]:
# Per-person message, media and deleted counts (plain dict, no Counter)
msg_counts     = {p: 0 for p in PARTICIPANTS}
media_counts   = {p: 0 for p in PARTICIPANTS}
deleted_counts = {p: 0 for p in PARTICIPANTS}

for m in normal_msgs:
    msg_counts[m["sender"]] += 1
for m in messages:
    if m["type"] == "media":
        media_counts[m["sender"]] += 1
    elif m["type"] == "deleted":
        deleted_counts[m["sender"]] += 1

total_msgs         = sum(msg_counts.values())
sorted_by_activity = sorted(PARTICIPANTS, key=lambda p: -msg_counts[p])

print("  Per-person breakdown:")
print(f"  {'Name':<10} {'Messages':>8} {'Media':>6} {'Deleted':>8}")
print("  " + "-"*36)
for p in sorted_by_activity:
    print(f"  {p:<10} {msg_counts[p]:>8} {media_counts[p]:>6} {deleted_counts[p]:>8}")


  Per-person breakdown:
  Name       Messages  Media  Deleted
  ------------------------------------
  Rahul           940      7        6
  Priya           712      4        2
  Neha            624      8        3
  Aman            484      4        2
  Karan           345      7        2
  Vikas            22      2        0


## Feature 3 — Most Active Day and Hour

In [6]:
day_counts  = {}
hour_counts = {}
for m in normal_msgs:
    dk = m["dt"].strftime("%d/%m/%y")
    day_counts[dk]          = day_counts.get(dk, 0) + 1
    hour_counts[m["dt"].hour] = hour_counts.get(m["dt"].hour, 0) + 1

busiest_day_key   = max(day_counts,  key=lambda x: day_counts[x])
busiest_hour      = max(hour_counts, key=lambda x: hour_counts[x])
busiest_day_dt    = datetime.strptime(busiest_day_key, "%d/%m/%y")
busiest_day_count = day_counts[busiest_day_key]

print(f"Busiest day  : {busiest_day_dt.strftime('%d %B %Y')}  ({busiest_day_count} messages)")
print(f"Busiest hour : {busiest_hour:02d}:00 – {(busiest_hour+1)%24:02d}:00  "
      f"({hour_counts[busiest_hour]} total messages across all days)")


Busiest day  : 04 May 2024  (74 messages)
Busiest hour : 18:00 – 19:00  (244 total messages across all days)


## Feature 4 — NumPy Activity Heatmap (6 × 24 matrix)

In [7]:
# Build the 6x24 matrix with np.zeros and integer indexing
heatmap = np.zeros((6, 24), dtype=int)
for m in normal_msgs:
    pidx = PERSON_IDX.get(m["sender"])
    if pidx is not None:
        heatmap[pidx, m["dt"].hour] += 1

# NumPy aggregations
row_totals    = heatmap.sum(axis=1)    # messages per person
col_totals    = heatmap.sum(axis=0)    # messages per hour (all people)
row_peak_hour = heatmap.argmax(axis=1) # peak hour per person

print("Row totals (messages per person):", dict(zip(PARTICIPANTS, row_totals.tolist())))
print("Peak hour per person:", dict(zip(PARTICIPANTS, row_peak_hour.tolist())))

SHADES = [". ", "░ ", "▒ ", "█ "]

def shade_cell(val, row_max):
    if row_max == 0: return SHADES[0]
    pct = val / row_max
    if pct == 0:      return SHADES[0]
    elif pct <= 0.25: return SHADES[1]
    elif pct <= 0.50: return SHADES[2]
    else:             return SHADES[3]

print()
print("  ACTIVITY HEATMAP (columns = hours 00–23)")
print("  " + "-"*62)
hdr = "  " + " "*10
for h in range(24):
    hdr += f"{h:02d}" if h % 3 == 0 else "  "
print(hdr)
for i, person in enumerate(PARTICIPANTS):
    row     = heatmap[i]
    row_max = int(row.max())
    cells   = "".join(shade_cell(int(v), row_max) for v in row)
    suffix  = "  ← NIGHT OWL" if person == "Aman" else ""
    print(f"  {person:<10}{cells}{suffix}")


Row totals (messages per person): {'Rahul': 940, 'Priya': 712, 'Aman': 484, 'Karan': 345, 'Neha': 624, 'Vikas': 22}
Peak hour per person: {'Rahul': 18, 'Priya': 9, 'Aman': 4, 'Karan': 12, 'Neha': 18, 'Vikas': 17}

  ACTIVITY HEATMAP (columns = hours 00–23)
  --------------------------------------------------------------
            00    03    06    09    12    15    18    21    
  Rahul     ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ █ ▒ ▒ █ █ ▒ █ █ ▒ █ █ █ 
  Priya     . . . . . . ░ ▒ █ █ █ █ █ █ █ ▒ ▒ █ █ █ █ ▒ ▒ ░ 
  Aman      █ █ █ █ █ . . . . . . . . . ░ ░ ░ ░ ░ ░ ░ ░ . █   ← NIGHT OWL
  Karan     . . . . . . . ░ ▒ ▒ █ ▒ █ █ █ █ █ █ █ █ █ ▒ ░ ░ 
  Neha      . . . . . ▒ ░ ░ █ █ █ ▒ █ █ ▒ ░ █ █ █ █ █ ▒ ▒ ▒ 
  Vikas     . . . . . . . ▒ █ ▒ ▒ . ▒ █ . ▒ ▒ █ █ █ ▒ ▒ ▒ █ 


## Feature 5 — Top Words (Group & Per-Person)

In [8]:
STOP_WORDS = {
    "i", "is", "the", "a", "and", "or", "to", "of", "in", "on", "for",
    "it", "this", "that", "you", "me", "my", "he", "she", "we", "they",
    "are", "was", "be", "been", "have", "has", "do", "did", "but", "so",
    "at", "by", "from", "with", "as", "not", "its", "an", "if", "what",
    "how", "when", "which", "who", "his", "her", "their", "our", "your",
    "will", "just", "up", "about", "am", "had", "one", "no", "also",
    "then", "him", "them", "all", "more", "were", "would", "could",
    "there", "than", "out", "even", "now", "like", "very", "into",
    "get", "got", "can", "go", "too", "because", "really", "some",
    "after", "before", "over", "back", "only", "going", "here",
    "should", "know", "think", "come", "still", "see", "way",
    "these", "any", "us", "same", "most", "being", "new", "said",
}
PUNCT = '.,!?;:"()-[]{}@#&*^%$~`\\/'

def tokenize(text):
    words = []
    for w in text.lower().split():
        w = w.strip(PUNCT)
        if w and w not in STOP_WORDS and len(w) > 1:
            words.append(w)
    return words

def build_word_freq(msgs):
    freq = {}
    for m in msgs:
        for w in tokenize(m["text"]):
            freq[w] = freq.get(w, 0) + 1
    return freq

def make_bar(count, max_count, width=22):
    filled = round(count / max_count * width) if max_count else 0
    return "█" * filled + "░" * (width - filled)

global_freq  = build_word_freq(normal_msgs)
top10_global = sorted(global_freq.items(), key=lambda x: -x[1])[:10]

print("  THIS GROUP\'S FAVOURITE WORDS")
print("  " + "-"*46)
top_max = top10_global[0][1]
for word, count in top10_global:
    print(f"  {word:<14} {make_bar(count, top_max)}  {count}")

print()
print("  TOP 5 WORDS PER PERSON")
print("  " + "-"*46)
for p in PARTICIPANTS:
    p_msgs = [m for m in normal_msgs if m["sender"] == p]
    freq = build_word_freq(p_msgs)
    top5 = sorted(freq.items(), key=lambda x: -x[1])[:5]
    ws = "  |  ".join(f"{w}({c})" for w, c in top5)
    print(f"  {p:<8}: {ws}")


  THIS GROUP'S FAVOURITE WORDS
  ----------------------------------------------
  guys           ██████████████████████  318
  hai            ███████████████████░░░  268
  today          ██████████████████░░░░  257
  everyone       █████████████░░░░░░░░░  187
  telling        ████████████░░░░░░░░░░  179
  bhai           ███████████░░░░░░░░░░░  160
  started        ██████████░░░░░░░░░░░░  150
  scene          ██████████░░░░░░░░░░░░  145
  entire         ██████████░░░░░░░░░░░░  145
  please         ██████████░░░░░░░░░░░░  141

  TOP 5 WORDS PER PERSON
  ----------------------------------------------
  Rahul   : hai(263)  |  bhai(159)  |  scene(144)  |  kya(133)  |  yaar(105)
  Priya   : please(141)  |  everyone(137)  |  aman(93)  |  anyone(90)  |  okay(80)
  Aman    : sleep(71)  |  i'm(56)  |  anyone(49)  |  wonder(43)  |  night(41)
  Karan   : telling(179)  |  today(171)  |  started(150)  |  entire(145)  |  guys(120)
  Neha    : guys(101)  |  cant(58)  |  ok(52)  |  today(48)  |  wait(4

## Feature 6 — Response Times & Silent Streaks

In [9]:
def compute_response_times(messages):
    """
    Average response time per person.
    Gap = seconds between a message from person A and the NEXT message from person B.
    Gaps over 240 min excluded (not live responses).
    Uses datetime subtraction and timedelta.total_seconds().
    """
    timeline = [m for m in messages if m["type"] in ("normal", "media")]
    gaps = {p: [] for p in PARTICIPANTS}
    for i in range(1, len(timeline)):
        curr, prev = timeline[i], timeline[i-1]
        if curr["sender"] == prev["sender"]:
            continue
        gap_mins = (curr["dt"] - prev["dt"]).total_seconds() / 60
        if 0 < gap_mins <= 240:
            gaps[curr["sender"]].append(gap_mins)
    avg = {}
    for p in PARTICIPANTS:
        g = gaps[p]
        avg[p] = sum(g) / len(g) if g else float("inf")
    return avg

def compute_silent_streaks(normal_msgs, calendar):
    """
    Longest consecutive-day silent streak per person.
    Walks the calendar and counts runs of absent days.
    """
    results = {}
    for p in PARTICIPANTS:
        active = set(m["dt"].date() for m in normal_msgs if m["sender"] == p)
        max_streak, streak = 0, 0
        streak_start, best_start = None, None
        for d in calendar:
            if d not in active:
                if streak == 0: streak_start = d
                streak += 1
                if streak > max_streak:
                    max_streak = streak
                    best_start = streak_start
            else:
                streak = 0
        results[p] = (max_streak, best_start)
    return results

def fmt_mins(mins):
    if mins == float("inf"): return "N/A"
    if mins < 60: return f"{mins:.1f} min"
    return f"{mins/60:.1f} hrs"

avg_response = compute_response_times(messages)
silent_data  = compute_silent_streaks(normal_msgs, calendar)

valid_resp  = [(p, v) for p, v in avg_response.items() if v < float("inf")]
fastest_rep = min(valid_resp, key=lambda x: x[1])
slowest_rep = max(valid_resp, key=lambda x: x[1])

print("  RESPONSE PATTERNS")
print("  " + "-"*44)
for p in sorted(PARTICIPANTS, key=lambda x: avg_response[x]):
    print(f"  {p:<10}: {fmt_mins(avg_response[p])}")

print()
print("  LONGEST SILENT STREAKS")
print("  " + "-"*44)
for p in sorted(PARTICIPANTS, key=lambda x: -silent_data[x][0]):
    streak, s_start = silent_data[p]
    if streak == 0:
        print(f"  {p:<10}: 0 days (never went silent)")
    else:
        s_end = s_start + timedelta(days=streak - 1)
        print(f"  {p:<10}: {streak} days  "
              f"({s_start.strftime('%d %b')} – {s_end.strftime('%d %b')})")


  RESPONSE PATTERNS
  --------------------------------------------
  Vikas     : 34.0 min
  Karan     : 35.9 min
  Rahul     : 36.5 min
  Neha      : 37.7 min
  Priya     : 38.5 min
  Aman      : 54.3 min

  LONGEST SILENT STREAKS
  --------------------------------------------
  Vikas     : 11 days  (23 Apr – 03 May)
  Rahul     : 0 days (never went silent)
  Priya     : 0 days (never went silent)
  Aman      : 0 days (never went silent)
  Karan     : 0 days (never went silent)
  Neha      : 0 days (never went silent)


## Feature 7 — Personality Archetype Detection

In [10]:
# ── Scorer functions (one per archetype) ─────────────────────────────────

def score_spammer(person):
    bursts, cur_sender, cur_run = [], None, 0
    for m in normal_msgs:
        if m["sender"] == cur_sender:
            cur_run += 1
        else:
            if cur_sender == person and cur_run:
                bursts.append(cur_run)
            cur_sender, cur_run = m["sender"], 1
    if cur_sender == person and cur_run:
        bursts.append(cur_run)
    return sum(bursts) / len(bursts) if bursts else 0

def score_group_mom(person):
    caring = ["okay", "safe", "eat", "sleep", "take care", "are you", "please",
              "reminder", "drink water", "don't forget"]
    total = 0
    for m in normal_msgs:
        if m["sender"] != person: continue
        t = m["text"].lower()
        for kw in caring:
            total += t.count(kw)
    return total

def score_night_owl(person):
    pidx       = PERSON_IDX[person]
    night_cols = list(range(23, 24)) + list(range(0, 5))
    # NumPy fancy indexing on the heatmap built in Feature 4
    night_msgs = int(heatmap[pidx, night_cols].sum())
    row_total  = int(heatmap[pidx].sum())
    return night_msgs / row_total * 100 if row_total else 0

def score_storyteller(person):
    p_msgs = [m for m in normal_msgs if m["sender"] == person]
    if not p_msgs: return 0
    return sum(len(m["text"].split()) for m in p_msgs) / len(p_msgs)

def score_drama_queen(person):
    p_msgs = [m for m in normal_msgs if m["sender"] == person]
    if not p_msgs: return 0
    hits = sum(1 for m in p_msgs
               if (len(m["text"]) > 3 and m["text"].isupper())
               or m["text"].count("!") >= 2)
    return hits / len(p_msgs) * 100

def score_ghost(person):
    active = len(set(m["dt"].date() for m in normal_msgs if m["sender"] == person))
    return (total_days - active) / total_days * 100

def score_comedian(person):
    laugh  = {"lol", "lmao", "haha", "rofl", "lmfao", "hehe", "hahaha"}
    p_msgs = [m for m in normal_msgs if m["sender"] == person]
    if not p_msgs: return 0
    return sum(1 for m in p_msgs if any(lw in m["text"].lower() for lw in laugh)) / len(p_msgs) * 100

def score_question_master(person):
    p_msgs = [m for m in normal_msgs if m["sender"] == person]
    if not p_msgs: return 0
    return sum(1 for m in p_msgs if m["text"].strip().endswith("?")) / len(p_msgs) * 100

# ── BONUS ARCHETYPE: THE LATE-NIGHT PHILOSOPHER ──────────────────────────
# Inspired by Indian hostel life: the 3 AM thinker who sends reflective
# messages while everyone else sleeps.
# Detection rule: % of messages sent between 00:00–03:59 that contain
# philosophical / introspective words.
PHILO_WORDS = {"wonder", "think", "life", "feel", "why", "sense", "real",
               "truth", "alone", "future", "dream", "exist", "matter",
               "maybe", "perhaps", "lost", "meaning", "deep", "mind",
               "time", "weird", "strange"}

def score_philosopher(person):
    late = [m for m in normal_msgs if m["sender"] == person and 0 <= m["dt"].hour <= 3]
    if not late: return 0
    hits = sum(1 for m in late if any(w in m["text"].lower().split() for w in PHILO_WORDS))
    return hits / len(late) * 100

# ── Build score table & assign archetypes ─────────────────────────────────
ARCHETYPE_ORDER = [
    "THE SPAMMER", "THE GROUP MOM", "THE NIGHT OWL", "THE STORYTELLER",
    "THE DRAMA QUEEN", "THE GHOST", "THE COMEDIAN", "THE QUESTION MASTER",
    "THE LATE-NIGHT PHILOSOPHER",
]
SCORERS = {
    "THE SPAMMER":                score_spammer,
    "THE GROUP MOM":              score_group_mom,
    "THE NIGHT OWL":              score_night_owl,
    "THE STORYTELLER":            score_storyteller,
    "THE DRAMA QUEEN":            score_drama_queen,
    "THE GHOST":                  score_ghost,
    "THE COMEDIAN":               score_comedian,
    "THE QUESTION MASTER":        score_question_master,
    "THE LATE-NIGHT PHILOSOPHER": score_philosopher,
}

all_scores = {p: {arch: SCORERS[arch](p) for arch in ARCHETYPE_ORDER} for p in PARTICIPANTS}

# Greedy exclusive assignment: in priority order, best un-assigned person wins.
# Tie-break: first in ARCHETYPE_ORDER (AI-assisted: Claude clarified this logic).
assignments  = {}
taken_people = set()
taken_archs  = set()

for arch in ARCHETYPE_ORDER:
    candidates = [(p, all_scores[p][arch]) for p in PARTICIPANTS if p not in taken_people]
    if not candidates: break
    best_p, best_s = max(candidates, key=lambda x: x[1])
    assignments[best_p] = (arch, best_s)
    taken_people.add(best_p); taken_archs.add(arch)

for p in PARTICIPANTS:
    if p not in assignments:
        remaining = [a for a in ARCHETYPE_ORDER if a not in taken_archs]
        best_a = max(remaining, key=lambda a: all_scores[p][a]) if remaining else ARCHETYPE_ORDER[-1]
        assignments[p] = (best_a, all_scores[p][best_a])
        taken_archs.add(best_a)

print("  RAW SCORES (selected columns):")
print(f"  {'Person':<10} {'Spammer':>8} {'Mom':>8} {'NightOwl':>9} {'Story':>7} {'Drama':>7} {'Ghost':>7}")
print("  " + "-"*64)
for p in PARTICIPANTS:
    s = all_scores[p]
    print(f"  {p:<10} {s['THE SPAMMER']:>8.1f} {s['THE GROUP MOM']:>8.1f} "
          f"{s['THE NIGHT OWL']:>9.1f} {s['THE STORYTELLER']:>7.1f} "
          f"{s['THE DRAMA QUEEN']:>7.1f} {s['THE GHOST']:>7.1f}")
print()
print("  ASSIGNED ARCHETYPES:")
for p in PARTICIPANTS:
    arch, score = assignments[p]
    print(f"  {p:<8}  →  {arch}  (score: {score:.1f})")


  RAW SCORES (selected columns):
  Person      Spammer      Mom  NightOwl   Story   Drama   Ghost
  ----------------------------------------------------------------
  Rahul           4.5      0.0      13.5     2.6     0.0     0.0
  Priya           1.7    621.0       1.3     5.0     0.0     0.0
  Aman            2.8     99.0      80.4     5.0     0.0     0.0
  Karan           1.2     29.0       2.0    57.0     0.0     0.0
  Neha            2.5     23.0       4.8     5.3    63.3     0.0
  Vikas           1.0      0.0       9.1     1.8     0.0    73.3

  ASSIGNED ARCHETYPES:
  Rahul     →  THE SPAMMER  (score: 4.5)
  Priya     →  THE GROUP MOM  (score: 621.0)
  Aman      →  THE NIGHT OWL  (score: 80.4)
  Karan     →  THE STORYTELLER  (score: 57.0)
  Neha      →  THE DRAMA QUEEN  (score: 63.3)
  Vikas     →  THE GHOST  (score: 73.3)


## Feature 8 — The Final Report

In [11]:
W = 64

def divider(ch="="):
    return ch * W

def lpad(label, value, pad=22):
    return f"  {label:<{pad}}: {value}"

def bar_chart(count, max_count, width=20):
    if max_count == 0: return "░" * width
    filled = round(count / max_count * width)
    return "█" * filled + "░" * (width - filled)

def shade_cell(val, row_max):
    if row_max == 0: return ". "
    pct = val / row_max
    if pct == 0:      return ". "
    elif pct <= 0.25: return "░ "
    elif pct <= 0.50: return "▒ "
    else:             return "█ "

def arch_detail(person, arch, score):
    p_msgs = [m for m in normal_msgs if m["sender"] == person]
    active = len(set(m["dt"].date() for m in p_msgs))
    d = {
        "THE SPAMMER":                f"avg {score:.1f} msgs per burst",
        "THE GROUP MOM":              f"caring keyword score: {int(score)}",
        "THE NIGHT OWL":              f"{score:.1f}% messages between 23h–04h",
        "THE STORYTELLER":            f"avg {score:.1f} words per message",
        "THE DRAMA QUEEN":            f"{score:.1f}% ALL-CAPS or 2+ exclamation msgs",
        "THE GHOST":                  f"active only {active} of {total_days} days",
        "THE COMEDIAN":               f"{score:.1f}% messages with laughter words",
        "THE QUESTION MASTER":        f"{score:.1f}% messages end with ?",
        "THE LATE-NIGHT PHILOSOPHER": f"{score:.1f}% late-night msgs are reflective",
    }
    return d.get(arch, f"score {score:.1f}")

def print_report():
    out = []
    def P(s=""): out.append(s)

    P(divider())
    P("  GROUPDNA REPORT  —  \"Hostel Bois 4ever\"")
    P(f"  {total_days} days  •  {total_msgs:,} messages  •  {len(PARTICIPANTS)} members")
    P(divider())
    P()
    P(lpad("Period",
           f"{start_date.strftime('%d %B %Y')}  to  {end_date.strftime('%d %B %Y')}"))
    P(lpad("Skipped",
           f"{system_count} system  |  {media_count} media-omitted  |  {deleted_count} deleted"))
    P(lpad("Busiest day",
           f"{busiest_day_dt.strftime('%d %B %Y')}  ({busiest_day_count} messages)"))
    P(lpad("Busiest hour",
           f"{busiest_hour:02d}:00 – {(busiest_hour+1)%24:02d}:00  "
           f"({hour_counts[busiest_hour]} total messages)"))
    P()

    P(f"  {'MESSAGES PER PERSON':^{W-4}}")
    P("  " + "-"*(W-4))
    max_c = msg_counts[sorted_by_activity[0]]
    for p in sorted_by_activity:
        cnt = msg_counts[p]
        pct = cnt / total_msgs * 100
        P(f"  {p:<8}  {bar_chart(cnt, max_c, 20)}  {cnt:>4}  ({pct:>4.1f}%)")
    P()

    P(f"  {'ACTIVITY HEATMAP  (columns = hours 00–23)':^{W-4}}")
    P("  " + "-"*(W-4))
    hdr = "  " + " "*10
    for h in range(24):
        hdr += f"{h:02d}" if h % 3 == 0 else "  "
    P(hdr)
    for i, person in enumerate(PARTICIPANTS):
        row     = heatmap[i]
        row_max = int(row.max())
        cells   = "".join(shade_cell(int(v), row_max) for v in row)
        suffix  = "  ← NIGHT OWL" if assignments[person][0] == "THE NIGHT OWL" else ""
        P(f"  {person:<10}{cells}{suffix}")
    P()

    P(f"  {'THIS GROUP\'S FAVOURITE WORDS':^{W-4}}")
    P("  " + "-"*(W-4))
    top_max = top10_global[0][1]
    for word, count in top10_global:
        P(f"  {word:<14} {bar_chart(count, top_max, 22)}  {count}")
    P()

    P(f"  {'RESPONSE PATTERNS':^{W-4}}")
    P("  " + "-"*(W-4))
    P(f"  {'Fastest replier':<22}: {fastest_rep[0]}  (avg {fmt_mins(fastest_rep[1])})")
    P(f"  {'Slowest replier':<22}: {slowest_rep[0]}  (avg {fmt_mins(slowest_rep[1])})")
    P()

    P(f"  {'LONGEST SILENT STREAKS  (consecutive days offline)':^{W-4}}")
    P("  " + "-"*(W-4))
    for p in sorted(PARTICIPANTS, key=lambda x: -silent_data[x][0]):
        streak, s_start = silent_data[p]
        if streak == 0:
            P(f"  {p:<10}: 0 days  (never went silent)")
        else:
            s_end = s_start + timedelta(days=streak - 1)
            P(f"  {p:<10}: {streak} days  "
              f"({s_start.strftime('%d %b')} – {s_end.strftime('%d %b')})")
    P()

    P(f"  {'PERSONALITY ARCHETYPES':^{W-4}}")
    P("  " + "-"*(W-4))
    for p in PARTICIPANTS:
        arch, score = assignments[p]
        detail = arch_detail(p, arch, score)
        P(f"  {p:<8}  →  {arch:<32} ({detail})")
    P()
    P(f"  {'★ BONUS: THE LATE-NIGHT PHILOSOPHER':^{W-4}}")
    P(f"  {'Detection: % of 00–03h messages containing reflective words':^{W-4}}")
    P(f"  {'(wonder, life, exist, meaning, alone, future …)':^{W-4}}")
    P()

    P(divider())
    P("  Generated by GroupDNA  •  Python + NumPy  •  No pandas / matplotlib".center(W))
    P(divider())
    return "\n".join(out)


report = print_report()
print(report)


  GROUPDNA REPORT  —  "Hostel Bois 4ever"
  60 days  •  3,127 messages  •  6 members

  Period                : 01 April 2024  to  30 May 2024
  Skipped               : 4 system  |  32 media-omitted  |  15 deleted
  Busiest day           : 04 May 2024  (74 messages)
  Busiest hour          : 18:00 – 19:00  (244 total messages)

                      MESSAGES PER PERSON                     
  ------------------------------------------------------------
  Rahul     ████████████████████   940  (30.1%)
  Priya     ███████████████░░░░░   712  (22.8%)
  Neha      █████████████░░░░░░░   624  (20.0%)
  Aman      ██████████░░░░░░░░░░   484  (15.5%)
  Karan     ███████░░░░░░░░░░░░░   345  (11.0%)
  Vikas     ░░░░░░░░░░░░░░░░░░░░    22  ( 0.7%)

           ACTIVITY HEATMAP  (columns = hours 00–23)          
  ------------------------------------------------------------
            00    03    06    09    12    15    18    21    
  Rahul     ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ █ ▒ ▒ █ █ ▒ █ █ ▒ █ █ █ 
  Priya

## Reflection

**Hardest part:** Feature 6 — response times. Deciding to cap gaps at 240 minutes, handling same-sender skips, and getting `timedelta.total_seconds()` right took the most debugging.

**What I'd do differently:** Normalise all archetype scores to a 0–100 scale before comparing so mixed-unit comparisons (e.g., burst count vs keyword count) are more principled.

**Key insight:** Building a word-frequency counter from a plain `dict` instead of `collections.Counter`, and a terminal heatmap instead of matplotlib, teaches you *what the library is actually doing underneath*.

**My archetype (run on my own chat):  THE NIGHT OWL**

---
*"I did all of this with just Week 1 Python."*